# Hadamard

In [ ]:
from qualtran import Bloq, CompositeBloq, BloqBuilder, Signature, Register
from qualtran import QBit, QInt, QUInt, QAny
from qualtran.drawing import show_bloq, show_call_graph, show_counts_sigma
from typing import *
import numpy as np
import sympy
import cirq

## `Hadamard`
The Hadamard gate

This converts between the X and Z basis.

$$
\begin{aligned}
H |0\rangle = |+\rangle \\
H |-\rangle = |1\rangle
\end{aligned}
$$

#### Registers
 - `q`: The qubit


In [ ]:
from qualtran.bloqs.basic_gates import Hadamard

### Example Instances

In [ ]:
hadamard = Hadamard()

#### Graphical Signature

In [ ]:
from qualtran.drawing import show_bloqs
show_bloqs([hadamard],
           ['`hadamard`'])

## `CHadamard`
The controlled Hadamard gate

#### Registers
 - `ctrl`: The control qubit.
 - `target`: The target qubit.


In [ ]:
from qualtran.bloqs.basic_gates import CHadamard

### Example Instances

In [ ]:
chadamard = Hadamard().controlled()
assert isinstance(chadamard, CHadamard)

#### Graphical Signature

In [ ]:
from qualtran.drawing import show_bloqs
show_bloqs([chadamard],
           ['`chadamard`'])

In [ ]:
show_bloq(chadamard, 'musical_score')

### Specialty circuits

The `CHadamard` bloq is atomic and cannot be decomposed with `.decompose_bloq()`. An actual implementation on an error-corrected quantum computer will likely be architecture-dependent. A naive circuit for CHadamard can be found using Cirq.

In [ ]:
circuit = cirq.Circuit(cirq.decompose_multi_controlled_rotation(
    cirq.unitary(cirq.H),
    controls=[cirq.NamedQubit('ctrl')],
    target=cirq.NamedQubit('q'),
))
circuit

## Quantum vs. Classical probability

The Hadamard gate sets up an equal superposition between the $|0\rangle$ and $|1\rangle$ states. If we simulate this simple circuit, we see that the probability *amplitudes* of the two states are equal, so if we were to measure the state we'd have an equal chance of measuring `0` and `1`.

In [ ]:
from qualtran.bloqs.basic_gates import ZeroState

# Simple circuit that uses Hadamard to set up
# an equal superposition between |0> and |1>
bb = BloqBuilder()
q = bb.add(ZeroState())
q = bb.add(Hadamard(), q=q)

h_circuit = bb.finalize(q=q)
show_bloq(h_circuit, 'musical_score')
print(h_circuit.tensor_contract())

If we do a subsequent Hadamard gate on the equal superposition, the amplitudes (which have signs and phases in addition to magnitude) interfere and we are guaranteed to measure `0`.

In [ ]:
# Set of up an equal superposition with Hadamard,
# undo it with a subsequent Hadamard.
# The resulting probability amplitudes give 100% change of |0>
bb = BloqBuilder()
q = bb.add(ZeroState())
q = bb.add(Hadamard(), q=q)
q = bb.add(Hadamard(), q=q)

hh_circuit = bb.finalize(q=q)
show_bloq(hh_circuit, 'musical_score')
print(hh_circuit.tensor_contract().round(4))

This is somewhat remarkable. Classically, you can't un-flip a coin. We can attempt to "peek" at the system in the middle of the circuit by using a `MeasZ` and then proceeding as before.

In [ ]:
from qualtran.bloqs.basic_gates.z_basis import MeasZ
from qualtran.bloqs.bookkeeping import Cast
from qualtran import CBit

bb = BloqBuilder()
q = bb.add(ZeroState())
q = bb.add(Hadamard(), q=q)
c = bb.add(MeasZ(), q=q)

q = bb.add(Cast(CBit(), QBit()), reg=c)
q = bb.add(Hadamard(), q=q)

hmh_circuit = bb.finalize(q=q)
show_bloq(hmh_circuit, 'musical_score')

We have to use a more complicated simulation proceedure to account for the mixture of classical and quantum operations. Instead of getting a vector of *amplitudes* we get a density matrix with probabilities along the diagonal and quantum couplings on the off diagonals.

In [ ]:
from qualtran.simulation.tensor._quimb import cbloq_to_superquimb

print("Density matrix -H-")
print(cbloq_to_superquimb(h_circuit, friendly_indices=True).to_dense(('q_0rf',), ('q_0rb',)).round(4))

print("\nDensity matrix -H-H-")
print(cbloq_to_superquimb(hh_circuit, friendly_indices=True).to_dense(('q_0rf',), ('q_0rb',)).round(4))

print("\nDensity matrix -H-Mz-H-")
print(cbloq_to_superquimb(hmh_circuit, friendly_indices=True).to_dense(('q_0rf',), ('q_0rb',)).round(4))